<a href="https://colab.research.google.com/github/R1BkAr0n/project/blob/main/%D0%A4%D0%B8%D0%BD%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9_%D1%8D%D1%82%D0%B0%D0%BF_%D0%BF%D1%80%D0%BE%D0%B5%D0%BA%D1%82%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install python-telegram-bot==20.7 nest_asyncio numpy matplotlib --quiet
print("✅ Установлено!")

✅ Установлено!


In [2]:
import asyncio
import nest_asyncio
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
from telegram import Update, ReplyKeyboardMarkup, InlineKeyboardMarkup, InlineKeyboardButton
from telegram.ext import Application, CommandHandler, MessageHandler, CallbackQueryHandler, filters, ContextTypes

nest_asyncio.apply()

TOKEN = '8796827162:AAHnT3viOhi5SQbHBRn8uQnfhy9tiuVEwn0'
G_THEORY = 9.81

MAIN_KB = ReplyKeyboardMarkup([
    ["📋 Эксперименты", "⚙️ Калибровка"],
    ["📊 Свой график", "❓ Помощь"]
], resize_keyboard=True)

BACK_KB = ReplyKeyboardMarkup([["🔙 Меню"]], resize_keyboard=True)

user_mode = {}
user_graph_data = {}

EXPERIMENT_INFO = {
    "g": (
        "*🪨 Свободное падение тел*\n\n"
        "*🎯 Цель опыта:* Измерить ускорение свободного падения g\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*📚 Теория*\n"
        "Когда тело падает свободно (без сопротивления воздуха), "
        "оно движется равноускоренно с ускорением g ≈ 9.81 м/с².\n\n"
        "Формула высоты: *h = g·t²/2*\n"
        "• h — высота падения (метры)\n"
        "• t — время падения (секунды)\n"
        "• g — ускорение свободного падения (м/с²)\n\n"
        "Отсюда: *g = 2h/t²* — для каждого измерения\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*📐 Что такое линеаризация?*\n"
        "Формула h = g·t²/2 — это *парабола* (квадратичная зависимость).\n"
        "Параболу сложно анализировать глазами — не видно точного значения g.\n\n"
        "Умножаем обе части на 2:\n"
        "• *2h = g·t²*\n\n"
        "Теперь если по оси X отложить t² (время в квадрате),\n"
        "а по оси Y — 2h (удвоенную высоту),\n"
        "то получится *прямая линия* с наклоном g!\n\n"
        "Это и есть *линеаризация* — превращение сложной кривой в простую прямую 🔄\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*🔬 Оборудование:*\n"
        "• Линейка или рулетка\n"
        "• Секундомер (можно в телефоне)\n"
        "• Небольшой тяжёлый предмет (ластик, монета, ключ)\n"
        "• Стул или стол для создания высоты\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*📝 Порядок выполнения:*\n"
        "1️⃣ Измерь высоту h в метрах (например, 0.5 м = 50 см)\n"
        "2️⃣ Брось предмет *без толчка* и одновременно запусти секундомер\n"
        "3️⃣ Останови секундомер точно в момент удара о пол\n"
        "4️⃣ Повтори с 3-5 разных высот (например, 0.5, 0.8, 1.0, 1.2, 1.5 м)\n"
        "5️⃣ Запиши все пары h и t\n\n"
        "*Ввод данных:* пары `h t` через пробел\n"
        "Пример: `0.5 0.32 1.0 0.45 1.5 0.55`\n\n"
        "Теоретическое значение: *g = 9.81 м/с²*"
    ),
    "pend": (
        "*⏰ Математический маятник*\n\n"
        "*🎯 Цель опыта:* Измерить g через колебания маятника\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*📚 Теория*\n"
        "Маятник — это груз на невесомой нити. Если его отклонить и отпустить, "
        "он начнёт колебаться. Время одного полного колебания туда-обратно "
        "называется *периодом T*.\n\n"
        "Формула периода: *T = 2π√(l/g)*\n"
        "• l — длина нити от подвеса до центра груза (м)\n"
        "• T — период одного колебания (с)\n"
        "• g — ускорение свободного падения\n\n"
        "Отсюда: *g = 4π²l/T²*\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*📐 Линеаризация в маятнике*\n"
        "Формула содержит квадратный корень — неудобно для графика!\n\n"
        "Возводим в квадрат:\n"
        "• *T² = (4π²/g)·l*\n\n"
        "Это *прямая* зависимость T² от l!\n"
        "• По X: длина нити l\n"
        "• По Y: квадрат периода T²\n"
        "• Наклон прямой = 4π²/g\n\n"
        "Проведи опыт с 3-4 разными длинами — и построй график!\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*🔬 Оборудование:*\n"
        "• Прочная нитка (50-100 см)\n"
        "• Небольшой груз (гайка, болт, шарик пластилина)\n"
        "• Линейка для измерения длины\n"
        "• Секундомер (в телефоне)\n"
        "• Штатив или перекладина для подвеса\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*📝 Порядок выполнения:*\n"
        "1️⃣ Измерь длину l от точки подвеса до *центра груза* (в метрах!)\n"
        "2️⃣ Отклони маятник на *10-15°* (не больше!)\n"
        "3️⃣ Отпусти и засеки время *10 полных колебаний*\n"
        "4️⃣ Раздели это время на 10 — получишь период T одного колебания\n"
        "5️⃣ Повтори 3 раза с той же длиной для точности\n\n"
        "*Ввод данных:* длину l, потом 3-5 значений периода T\n"
        "Пример: `0.5 1.42 1.40 1.43`\n\n"
        "💡 *Для графика* измерь период с 3-4 *разными* длинами нити!"
    ),
    "lens": (
        "*🔍 Оптика — собирающая линза*\n\n"
        "*🎯 Цель опыта:* Найти фокусное расстояние линзы f\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*📚 Теория*\n"
        "Линза преломляет свет. Если поставить предмет (свечу) с одной стороны, "
        "с другой появится его изображение на определённом расстоянии.\n\n"
        "Когда изображение *резкое и чёткое* — экран находится в фокусе.\n\n"
        "Формула тонкой линзы: *1/f = 1/u + 1/v*\n"
        "• u — расстояние от предмета до линзы (см)\n"
        "• v — расстояние от линзы до экрана (см)\n"
        "• f — фокусное расстояние (см)\n\n"
        "Отсюда: *f = (u·v)/(u+v)*\n\n"
        "Оптическая сила: *D = 1/f* (в диоптриях, f в метрах)\n"
        "Например: f = 20 см = 0.2 м → D = 1/0.2 = 5 дптр\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*📐 Линеаризация в оптике*\n"
        "Формула 1/f = 1/u + 1/v — гиперболическая, неудобная для графика.\n\n"
        "Умножаем на u·v·f:\n"
        "• *u·v = f·(u+v)*\n\n"
        "Теперь это прямая!\n"
        "• По X: (u+v) — сумма расстояний\n"
        "• По Y: (u·v) — произведение расстояний\n"
        "• Наклон прямой = f!\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*🔬 Оборудование:*\n"
        "• Собирающая линза (лупа, +2...+5 дптр)\n"
        "• Белый экран (лист бумаги)\n"
        "• Линейка 30-50 см\n"
        "• Свеча или настольная лампа\n"
        "• Затемнённое помещение (чем темнее, тем лучше)\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*📝 Порядок выполнения (ВАЖНО — читай внимательно!):*\n\n"
        "*Шаг 1:* Поставь свечу на расстоянии примерно 20-25 см от линзы\n\n"
        "*Шаг 2:* С другой стороны линзы поставь экран и *медленно двигай его*, "
        "пока не увидишь *самое чёткое и резкое* изображение пламени\n"
        "(изображение будет перевёрнутым — это нормально!)\n\n"
        "*Шаг 3:* *Зафиксируй экран* в этом положении — не двигай!\n\n"
        "*Шаг 4:* Теперь аккуратно измерь:\n"
        "• u — расстояние от свечи до центра линзы (см)\n"
        "• v — расстояние от центра линзы до экрана (см)\n\n"
        "*Шаг 5:* Повтори опыт с другими расстояниями:\n"
        "Поставь свечу дальше (25, 30, 35, 40 см) и каждый раз "
        "заново лови чёткое изображение!\n\n"
        "⚠ *Почему нельзя мерить расстояние сразу?*\n"
        "Потому что фокус находится опытным путём! Сначала двигаешь "
        "экран туда-сюда, находишь самое резкое изображение, "
        "и только потом измеряешь расстояния u и v.\n\n"
        "*Ввод данных:* пары `u v` через пробел\n"
        "Пример: `20 30 25 28 30 27 35 26`\n\n"
        "💡 *Совет:* Двигай экран медленно — разница в 2-3 мм меняет чёткость!"
    ),
    "dens": (
        "*💧 Измерение плотности вещества*\n\n"
        "*🎯 Плотность — это масса в единице объёма*\n"
        "ρ = m/V (г/см³ или кг/м³)\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*📐 Линеаризация для плотности*\n"
        "Формула: *m = ρ·V* — уже прямая линия!\n"
        "• По X: объём V\n"
        "• По Y: масса m\n"
        "• Наклон прямой = плотность ρ\n\n"
        "Измерь несколько образцов — я построю график и найду ρ!\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*Выбери метод измерения:*"
    ),
}

DENSITY_INFO = {
    "d_f": (
        "*📦 Метод 1: По форме (прямоугольный предмет)*\n\n"
        "*Когда использовать:* Предмет имеет правильную форму — брусок, "
        "книга, коробка, кусок пенопласта.\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*📝 Порядок действий:*\n"
        "1️⃣ Взвесь предмет — масса m (в граммах)\n"
        "2️⃣ Измерь длину l (см), ширину w (см), высоту h (см)\n"
        "3️⃣ Вычисли объём: V = l·w·h (см³)\n"
        "4️⃣ Плотность: ρ = m/(l·w·h) г/см³\n\n"
        "*Ввод:* `m l w h` (можно несколько образцов)\n"
        "Пример: `50 5 2 1 80 6 3 1.5`\n\n"
        "Справочные значения: Дерево 0.5-0.8, Алюминий 2.7, Железо 7.8 г/см³"
    ),
    "d_v": (
        "*💧 Метод 2: По объёму вытесненной воды (Архимед)*\n\n"
        "*Когда использовать:* Предмет неправильной формы — камень, болт, "
        "статуэтка, картофелина.\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*📝 Порядок действий:*\n"
        "1️⃣ Взвесь предмет в воздухе — масса m (г)\n"
        "2️⃣ Налей воду в мензурку, запиши начальный объём Vi (мл)\n"
        "3️⃣ Осторожно опусти предмет в воду (он должен утонуть)\n"
        "4️⃣ Запиши конечный объём Vf (мл)\n"
        "5️⃣ Объём тела: V = Vf - Vi (1 мл = 1 см³)\n"
        "6️⃣ Плотность: ρ = m/(Vf-Vi) г/см³\n\n"
        "*Ввод:* `m Vi Vf`\n"
        "Пример: `60 100 120 80 100 130`\n\n"
        "💡 Предмет должен быть полностью погружён в воду!"
    ),
    "d_l": (
        "*🧪 Метод 3: Плотность жидкости*\n\n"
        "*Когда использовать:* Нужно узнать плотность неизвестной жидкости — "
        "молока, масла, солёной воды, сиропа.\n\n"
        "━━━━━━━━━━━━━━━━━━━━\n"
        "*📝 Порядок действий:*\n"
        "1️⃣ Взвесь пустую мензурку — масса mc (г)\n"
        "2️⃣ Налей жидкость, взвесь снова — масса ml (г)\n"
        "3️⃣ Масса жидкости: m = ml - mc\n"
        "4️⃣ Посмотри объём жидкости vl на шкале мензурки (мл)\n"
        "5️⃣ Плотность: ρ = (ml-mc)/vl г/см³\n\n"
        "*Ввод:* `mc ml vl`\n"
        "Пример: `50 150 100 50 140 90`\n\n"
        "Справочные: Вода 1.0, Масло 0.9, Молоко 1.03, Спирт 0.8 г/см³"
    ),
}

print("✅ Настройки готовы!")

✅ Настройки готовы!


In [3]:
def deviation(v, t):
    if t == 0: return 0, "?"
    p = abs(v - t) / t * 100
    if p < 5: return p, "✅ Отлично!"
    elif p < 10: return p, "👍 Хорошо"
    return p, "🔄 Можно точнее!"

print("✅ Функции готовы!")

✅ Функции готовы!


In [4]:
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_mode[update.effective_user.id] = "menu"
    await update.message.reply_text(
        "🧪 *Привет! Я твой помощник-лаборант по физике!*\n\n"
        "Я помогу тебе провести настоящие научные опыты и понять, как работают физические законы.\n\n"
        "🎯 *Что я умею:*\n"
        "📋 Проводить эксперименты с графиками\n"
        "⚙️ Учить правильно измерять\n"
        "📊 Строить графики по любым данным\n"
        "📐 Объяснять, как превратить сложную формулу в прямую линию (линеаризация)\n\n"
        "*С чего начнём?*",
        reply_markup=MAIN_KB, parse_mode='Markdown'
    )

async def menu(update: Update, context: ContextTypes.DEFAULT_TYPE):
    uid = update.effective_user.id
    t = update.message.text

    if t == "🔙 Меню":
        user_mode[uid] = "menu"
        await update.message.reply_text("*🏠 Главное меню*\nВыбери, что будем делать:", reply_markup=MAIN_KB, parse_mode='Markdown')
        return

    if t == "📋 Эксперименты":
        kb = [
            [InlineKeyboardButton("🪨 Свободное падение", callback_data="g")],
            [InlineKeyboardButton("⏰ Маятник", callback_data="pend")],
            [InlineKeyboardButton("🔍 Оптика (линза)", callback_data="lens")],
            [InlineKeyboardButton("💧 Плотность", callback_data="dens")]
        ]
        await update.message.reply_text(
            "*📋 Выбери эксперимент:*\n\n"
            "В каждом опыте ты:\n"
            "• Узнаешь физическую теорию\n"
            "• Научишься правильно измерять\n"
            "• Построишь график и поймёшь его смысл\n"
            "• Освоишь *линеаризацию* 📐",
            reply_markup=InlineKeyboardMarkup(kb), parse_mode='Markdown'
        )
        return

    if t == "⚙️ Калибровка":
        user_mode[uid] = "menu"
        await update.message.reply_text(
            "*⚙️ Искусство точных измерений*\n\n"
            "🔁 *1. Повторяй измерения 3-5 раз*\n"
            "Случайные ошибки уменьшатся, результат станет надёжнее.\n\n"
            "📝 *2. Записывай всё сразу*\n"
            "Память подводит! Держи таблицу.\n\n"
            "🔧 *3. Проверяй приборы до опыта*\n"
            "Линейка с нуля? Секундомер обнулён? Весы на ровной поверхности?\n\n"
            "👁 *4. Избегай параллакса*\n"
            "Смотри на шкалу строго перпендикулярно, на уровне глаз.\n\n"
            "⚖ *5. Устойчивость установки*\n"
            "Штатив не шатается, груз висит свободно.\n\n"
            "📏 *6. Выбирай правильный диапазон*\n"
            "Для падения — высота от 0.5 м, для маятника — длина от 0.3 м.",
            reply_markup=MAIN_KB, parse_mode='Markdown'
        )
        return

    if t == "📊 Свой график":
        user_mode[uid] = "custom_graph"
        user_graph_data[uid] = {'x': [], 'y': []}
        await update.message.reply_text("*📊 Вводи пары x y*\nНапиши `Готово` для графика", reply_markup=BACK_KB, parse_mode='Markdown')
        return

    if t == "❓ Помощь":
        user_mode[uid] = "menu"
        await update.message.reply_text(
            "*❓ Числа через пробел!*\n✅ `10 20`\n❌ `10,20`\n\n"
            "В каждом опыте читай теорию — там объяснена линеаризация!",
            reply_markup=MAIN_KB, parse_mode='Markdown'
        )
        return

print("✅ Меню готово!")

✅ Меню готово!


In [5]:
async def exp_info(update: Update, context: ContextTypes.DEFAULT_TYPE):
    q = update.callback_query
    await q.answer()
    uid = update.effective_user.id
    d = q.data

    if d == "g":
        user_mode[uid] = "g"
        await q.edit_message_text(EXPERIMENT_INFO["g"], parse_mode='Markdown')
        await q.message.reply_text("✏️ *Вводи пары h t:*", reply_markup=BACK_KB, parse_mode='Markdown')

    elif d == "pend":
        user_mode[uid] = "pend"
        await q.edit_message_text(EXPERIMENT_INFO["pend"], parse_mode='Markdown')
        await q.message.reply_text("✏️ *Вводи l T1 T2 T3:*", reply_markup=BACK_KB, parse_mode='Markdown')

    elif d == "lens":
        user_mode[uid] = "lens"
        await q.edit_message_text(EXPERIMENT_INFO["lens"], parse_mode='Markdown')
        await q.message.reply_text("✏️ *Вводи пары u v:*", reply_markup=BACK_KB, parse_mode='Markdown')

    elif d == "dens":
        user_mode[uid] = "dens_choice"  # ждём выбора метода
        kb = [
            [InlineKeyboardButton("📦 По форме (m,l,w,h)", callback_data="d_f")],
            [InlineKeyboardButton("💧 По объёму (m,Vi,Vf)", callback_data="d_v")],
            [InlineKeyboardButton("🧪 Жидкость (mc,ml,vl)", callback_data="d_l")]
        ]
        await q.edit_message_text(EXPERIMENT_INFO["dens"], reply_markup=InlineKeyboardMarkup(kb), parse_mode='Markdown')

async def dens_info(update: Update, context: ContextTypes.DEFAULT_TYPE):
    q = update.callback_query
    await q.answer()
    uid = update.effective_user.id
    d = q.data

    if d == "d_f":
        user_mode[uid] = "d_f"
        await q.edit_message_text(DENSITY_INFO["d_f"], parse_mode='Markdown')
        await q.message.reply_text("✏️ *Вводи m l w h:*", reply_markup=BACK_KB, parse_mode='Markdown')

    elif d == "d_v":
        user_mode[uid] = "d_v"
        await q.edit_message_text(DENSITY_INFO["d_v"], parse_mode='Markdown')
        await q.message.reply_text("✏️ *Вводи m Vi Vf:*", reply_markup=BACK_KB, parse_mode='Markdown')

    elif d == "d_l":
        user_mode[uid] = "d_l"
        await q.edit_message_text(DENSITY_INFO["d_l"], parse_mode='Markdown')
        await q.message.reply_text("✏️ *Вводи mc ml vl:*", reply_markup=BACK_KB, parse_mode='Markdown')

print("✅ Инструкции готовы!")

✅ Инструкции готовы!


In [6]:
async def handle_msg(update: Update, context: ContextTypes.DEFAULT_TYPE):
    uid = update.effective_user.id
    txt = update.message.text.strip()

    MENU_BUTTONS = ["🔙 Меню", "📋 Эксперименты", "⚙️ Калибровка", "📊 Свой график", "❓ Помощь"]
    if txt in MENU_BUTTONS:
        user_mode[uid] = "menu"
        if txt == "🔙 Меню":
            await update.message.reply_text("*🏠 Главное меню*", reply_markup=MAIN_KB, parse_mode='Markdown')
        else:
            await menu(update, context)
        return

    mode = user_mode.get(uid, "menu")

    if mode == "menu" or mode == "dens_choice" or mode not in ["g", "pend", "lens", "d_f", "d_v", "d_l", "custom_graph"]:
        await menu(update, context)
        return

    if mode == "custom_graph":
        await custom_graph(update, context)
        return

    try:
        nums = [float(x) for x in txt.replace(',', ' ').split()]
    except:
        await update.message.reply_text("❌ Введи числа через пробел!\nИли нажми `🔙 Меню`", parse_mode='Markdown')
        return

    # 🪨 Падение
    if mode == "g":
        if len(nums) % 2 != 0:
            await update.message.reply_text("❌ Нужны пары h t\nПример: `0.5 0.32 1.0 0.45`"); return

        h, t = nums[::2], nums[1::2]
        if any(v <= 0 for v in h + t):
            await update.message.reply_text("❌ Все числа должны быть положительными!"); return

        errors = []
        for hi, ti in zip(h, t):
            if hi > 100: errors.append(f"📏 Высота {hi} м — небоскрёб! Измеряй в метрах, не в сантиметрах.")
            elif hi > 10: errors.append(f"📏 Высота {hi} м — это 3-4 этажа! Возьми 0.3-2 м")
            elif hi > 3: errors.append(f"📏 Высота {hi} м — высоковато. Лучше 0.5-2 м")
            if hi < 0.1: errors.append(f"📏 Высота {hi} м = {hi*100} см — слишком мало. Возьми хотя бы 0.3 м")
            if ti > 5: errors.append(f"⏱ Время {ti}с — слишком долго. Проверь секундомер")
            if ti < 0.1: errors.append(f"⏱ Время {ti}с — слишком быстро. Увеличь высоту")
            t_theory = (2*hi/G_THEORY)**0.5
            if ti > t_theory * 3: errors.append(f"⚠ h={hi}м t={ti}с — время большое! Теория ~{t_theory:.2f}с. Не кидай вниз!")
            if ti < t_theory * 0.3: errors.append(f"⚠ h={hi}м t={ti}с — время маленькое! Теория ~{t_theory:.2f}с")

        if errors:
            await update.message.reply_text("⚠ *Данные нереалистичны:*\n\n" + "\n".join(errors[:3]) +
                                           "\n\nПроверь и введи заново!", parse_mode='Markdown'); return

        gv = [2*hi/ti**2 for hi, ti in zip(h, t)]
        ga, gs = np.mean(gv), np.std(gv)
        p, r = deviation(ga, G_THEORY)

        # Правильная аппроксимация: 2h = g·t² (прямая через ноль)
        t2 = np.array([x**2 for x in t])
        h2 = np.array([2*x for x in h])

        # Аппроксимация прямой Y = kX (без свободного члена)
        g_graph = np.sum(t2 * h2) / np.sum(t2 * t2)

        plt.figure(figsize=(10,6))
        plt.scatter(t2, h2, color='blue', s=100, edgecolors='black', zorder=5)
        xr = np.linspace(0, max(t2)*1.1, 100)
        plt.plot(xr, g_graph*xr, 'r--', lw=2, label=f'Аппроксимация: g={g_graph:.2f} м/с²')
        plt.plot(xr, G_THEORY*xr, 'g:', lw=2, label=f'Теория: g={G_THEORY} м/с²')
        plt.xlabel('t² (с²)'); plt.ylabel('2h (м)'); plt.title('Свободное падение: 2h = g·t²');
        plt.grid(True, alpha=0.3); plt.legend()
        bio = BytesIO(); plt.savefig(bio, format='png', dpi=150); bio.seek(0); plt.close()

        await update.message.reply_photo(bio, caption=f"*🪨 g = {ga:.2f}±{gs:.2f} м/с²*\nПо графику: g={g_graph:.2f} м/с²\nОтклонение {p:.1f}% {r}", parse_mode='Markdown')
        user_mode[uid] = "menu"
        await update.message.reply_text("✅ Готово!", reply_markup=MAIN_KB)

    # ⏰ Маятник
    elif mode == "pend":
        if len(nums) < 2: await update.message.reply_text("❌ Введи l T1 T2..."); return

        l, T = nums[0], nums[1:]
        if l <= 0 or any(x <= 0 for x in T): await update.message.reply_text("❌ Числа > 0!"); return

        errors = []
        if l > 3: errors.append(f"📏 Длина {l}м — гигантский маятник! Возьми 0.2-1.5м")
        if l < 0.1: errors.append(f"📏 Длина {l}м = {l*100}см — короткий. Минимум 20см")
        for ti in T:
            T_theory = 2*np.pi*(l/G_THEORY)**0.5
            if ti > T_theory*3: errors.append(f"⏱ Период {ti}с — большой для {l}м. Теория ~{T_theory:.2f}с")
            if ti < 0.5: errors.append(f"⏱ Период {ti}с — маленький. Это полное колебание (туда-обратно)?")

        if errors:
            await update.message.reply_text("⚠ *Данные нереалистичны:*\n\n" + "\n".join(errors[:3]) +
                                           "\n\nПроверь!", parse_mode='Markdown'); return

        gv = [4*np.pi**2*l/x**2 for x in T]
        ga, gs = np.mean(gv), np.std(gv)
        p, r = deviation(ga, G_THEORY)
        await update.message.reply_text(f"*⏰ g = {ga:.2f}±{gs:.2f} м/с²*\nОтклонение {p:.1f}% {r}", parse_mode='Markdown')
        user_mode[uid] = "menu"
        await update.message.reply_text("✅ Готово!", reply_markup=MAIN_KB)

    # 🔍 Линза
    elif mode == "lens":
        if len(nums) % 2 != 0: await update.message.reply_text("❌ Нужны пары u v\nПример: `20 30 25 28`"); return

        u, v = nums[::2], nums[1::2]
        if any(x<=0 or y<=0 for x,y in zip(u,v)): await update.message.reply_text("❌ Числа > 0!"); return

        errors = []
        for ui, vi in zip(u, v):
            if ui < 5: errors.append(f"📏 u={ui}см — близко к линзе. Минимум 10-15см")
            if vi < 5: errors.append(f"📏 v={vi}см — экран вплотную. Отодвинь!")
            if ui > 500: errors.append(f"📏 u={ui}см — 5 метров! Проверь единицы (см?)")

        if errors:
            await update.message.reply_text("⚠ *Данные нереалистичны:*\n\n" + "\n".join(errors[:3]) +
                                           "\n\nПроверь!", parse_mode='Markdown'); return

        fv = [(x*y)/(x+y) for x,y in zip(u,v)]
        fa, fs = np.mean(fv), np.std(fv)
        if fa > 100: await update.message.reply_text(f"⚠ f={fa:.0f}см — большое. Проверь см/мм", parse_mode='Markdown'); return
        if fa < 1: await update.message.reply_text(f"⚠ f={fa:.1f}см — маленькое. Проверь см/мм", parse_mode='Markdown'); return
        D = 100/fa

        # Правильная аппроксимация: uv = f·(u+v) (прямая через ноль)
        X = np.array([x+y for x,y in zip(u,v)])
        Y = np.array([x*y for x,y in zip(u,v)])
        f_graph = np.sum(X * Y) / np.sum(X * X)

        plt.figure(figsize=(10,6))
        plt.scatter(X, Y, color='blue', s=100, edgecolors='black', zorder=5)
        xr = np.linspace(0, max(X)*1.1, 100)
        plt.plot(xr, f_graph*xr, 'r--', lw=2, label=f'Аппроксимация: f={f_graph:.1f} см')
        plt.xlabel('u+v (см)'); plt.ylabel('u·v (см²)'); plt.title('Формула линзы: uv = f·(u+v)');
        plt.grid(True, alpha=0.3); plt.legend()
        bio = BytesIO(); plt.savefig(bio, format='png', dpi=150); bio.seek(0); plt.close()

        await update.message.reply_photo(bio, caption=f"*🔍 f = {fa:.1f}±{fs:.1f} см*\nПо графику: f={f_graph:.1f} см\nD = {D:.1f} дптр", parse_mode='Markdown')
        user_mode[uid] = "menu"
        await update.message.reply_text("✅ Готово!", reply_markup=MAIN_KB)

    # 💧 Плотность (форма)
    elif mode == "d_f":
        if len(nums) % 4 != 0: await update.message.reply_text("❌ Группы m l w h"); return

        rv, ms, vs, errors = [], [], [], []
        for i in range(0, len(nums), 4):
            m, l, w, h = nums[i:i+4]
            if any(x<=0 for x in [m,l,w,h]): await update.message.reply_text("❌ Числа > 0!"); return
            if m > 10000: errors.append(f"⚖ Масса {m}г — {m/1000}кг! Проверь весы")
            if l>100 or w>100 or h>100: errors.append(f"📏 {l}×{w}×{h}см — огромный предмет!")
            V = l*w*h; rho = m/V; rv.append(rho); ms.append(m); vs.append(V)
            if rho > 20: errors.append(f"⚠ Плотность {rho:.1f} г/см³ > золота!")
            if rho < 0.1: errors.append(f"⚠ Плотность {rho:.3f} г/см³ < воздуха!")

        if errors:
            await update.message.reply_text("⚠ *Данные нереалистичны:*\n\n" + "\n".join(errors[:3]) + "\n\nПроверь!", parse_mode='Markdown'); return

        ra, rs = np.mean(rv), np.std(rv)

        # График m = ρ·V (прямая через ноль)
        if len(ms) >= 2:
            V_arr = np.array(vs); m_arr = np.array(ms)
            rho_graph = np.sum(V_arr * m_arr) / np.sum(V_arr * V_arr)

            plt.figure(figsize=(10,6))
            plt.scatter(vs, ms, color='blue', s=100, edgecolors='black', zorder=5)
            xr = np.linspace(0, max(vs)*1.1, 100)
            plt.plot(xr, rho_graph*xr, 'r--', lw=2, label=f'ρ={rho_graph:.2f} г/см³')
            plt.xlabel('V (см³)'); plt.ylabel('m (г)'); plt.title('Плотность: m = ρ·V');
            plt.grid(True, alpha=0.3); plt.legend()
            bio = BytesIO(); plt.savefig(bio, format='png', dpi=150); bio.seek(0); plt.close()
            await update.message.reply_photo(bio, caption=f"*📦 ρ = {ra:.2f}±{rs:.2f} г/см³*\nПо графику: ρ={rho_graph:.2f} г/см³", parse_mode='Markdown')
        else:
            await update.message.reply_text(f"*📦 ρ = {ra:.2f} г/см³*\n(для графика нужно 2+ образца)", parse_mode='Markdown')
        user_mode[uid] = "menu"; await update.message.reply_text("✅ Готово!", reply_markup=MAIN_KB)

    # 💧 Плотность (объем)
    elif mode == "d_v":
        if len(nums) % 3 != 0: await update.message.reply_text("❌ Группы m Vi Vf"); return

        rv, ms, dV, errors = [], [], [], []
        for i in range(0, len(nums), 3):
            m, vi, vf = nums[i:i+3]
            if m<=0 or vi<=0 or vf<=vi: await update.message.reply_text("❌ Vf > Vi!"); return
            if vi>1000: errors.append(f"🧪 Vi={vi}мл — литр! Возьми меньше")
            if vf-vi<1: errors.append(f"📏 Объём {vf-vi:.2f}мл — мал. Погрешность большая")
            dv = vf-vi; rho = m/dv; rv.append(rho); ms.append(m); dV.append(dv)
            if rho>20: errors.append(f"⚠ ρ={rho:.1f} — не бывает!")

        if errors:
            await update.message.reply_text("⚠ *Данные нереалистичны:*\n\n" + "\n".join(errors[:3]) + "\n\nПроверь!", parse_mode='Markdown'); return

        ra, rs = np.mean(rv), np.std(rv)

        if len(ms) >= 2:
            dV_arr = np.array(dV); m_arr = np.array(ms)
            rho_graph = np.sum(dV_arr * m_arr) / np.sum(dV_arr * dV_arr)

            plt.figure(figsize=(10,6))
            plt.scatter(dV, ms, color='blue', s=100, edgecolors='black', zorder=5)
            xr = np.linspace(0, max(dV)*1.1, 100)
            plt.plot(xr, rho_graph*xr, 'r--', lw=2, label=f'ρ={rho_graph:.2f} г/см³')
            plt.xlabel('Vf-Vi (см³)'); plt.ylabel('m (г)'); plt.title('Плотность: m = ρ·(Vf-Vi)');
            plt.grid(True, alpha=0.3); plt.legend()
            bio = BytesIO(); plt.savefig(bio, format='png', dpi=150); bio.seek(0); plt.close()
            await update.message.reply_photo(bio, caption=f"*💧 ρ = {ra:.2f}±{rs:.2f} г/см³*\nПо графику: ρ={rho_graph:.2f} г/см³", parse_mode='Markdown')
        else:
            await update.message.reply_text(f"*💧 ρ = {ra:.2f} г/см³*", parse_mode='Markdown')
        user_mode[uid] = "menu"; await update.message.reply_text("✅ Готово!", reply_markup=MAIN_KB)

    # 💧 Плотность (жидкость)
    elif mode == "d_l":
        if len(nums) % 3 != 0: await update.message.reply_text("❌ Группы mc ml vl"); return

        rv, lm, vl_list, errors = [], [], [], []
        for i in range(0, len(nums), 3):
            mc, ml, v = nums[i:i+3]
            if mc<=0 or ml<=mc or v<=0: await update.message.reply_text("❌ ml > mc!"); return
            if v>1000: errors.append(f"🧪 Объём {v}мл — литр!")
            if v<10: errors.append(f"🧪 Объём {v}мл — мало")
            m = ml-mc; rho = m/v; rv.append(rho); lm.append(m); vl_list.append(v)
            if rho>15: errors.append(f"⚠ ρ={rho:.1f} — не жидкость!")

        if errors:
            await update.message.reply_text("⚠ *Данные нереалистичны:*\n\n" + "\n".join(errors[:3]) + "\n\nПроверь!", parse_mode='Markdown'); return

        ra, rs = np.mean(rv), np.std(rv)

        if len(lm) >= 2:
            V_arr = np.array(vl_list); m_arr = np.array(lm)
            rho_graph = np.sum(V_arr * m_arr) / np.sum(V_arr * V_arr)

            plt.figure(figsize=(10,6))
            plt.scatter(vl_list, lm, color='blue', s=100, edgecolors='black', zorder=5)
            xr = np.linspace(0, max(vl_list)*1.1, 100)
            plt.plot(xr, rho_graph*xr, 'r--', lw=2, label=f'ρ={rho_graph:.2f} г/см³')
            plt.xlabel('V (мл)'); plt.ylabel('m жидк (г)'); plt.title('Плотность жидкости: m = ρ·V');
            plt.grid(True, alpha=0.3); plt.legend()
            bio = BytesIO(); plt.savefig(bio, format='png', dpi=150); bio.seek(0); plt.close()
            await update.message.reply_photo(bio, caption=f"*🧪 ρ = {ra:.2f}±{rs:.2f} г/см³*\nПо графику: ρ={rho_graph:.2f} г/см³", parse_mode='Markdown')
        else:
            await update.message.reply_text(f"*🧪 ρ = {ra:.2f} г/см³*", parse_mode='Markdown')
        user_mode[uid] = "menu"; await update.message.reply_text("✅ Готово!", reply_markup=MAIN_KB)

print("✅ Расчёты готовы!")

✅ Расчёты готовы!


In [7]:
async def custom_graph(update: Update, context: ContextTypes.DEFAULT_TYPE):
    uid = update.effective_user.id
    txt = update.message.text.lower().strip()

    if txt == "готово":
        x = user_graph_data[uid]['x']
        y = user_graph_data[uid]['y']

        if len(x) < 2:
            await update.message.reply_text("❌ Минимум 2 точки для графика!")
            return

        # Строим график
        plt.figure(figsize=(10, 6))
        plt.scatter(x, y, color='blue', s=100, edgecolors='black', zorder=5, label=f'Точки ({len(x)} шт.)')

        xr = np.linspace(min(x)*0.9, max(x)*1.1, 100)

        # Линейный тренд
        z1 = np.polyfit(x, y, 1)
        k = z1[0]  # угловой коэффициент
        b = z1[1]  # свободный член
        plt.plot(xr, k*xr + b, 'r--', lw=2,
                label=f'Прямая: y={k:.2f}x+{b:.2f}')

        # Квадратичный тренд (если 3+ точки)
        if len(x) >= 3:
            z2 = np.polyfit(x, y, 2)
            a = z2[0]
            b2 = z2[1]
            c = z2[2]
            plt.plot(xr, a*xr**2 + b2*xr + c, 'g:', lw=2,
                    label=f'Парабола: y={a:.2f}x²+{b2:.2f}x+{c:.2f}')

        plt.xlabel('X', fontsize=12)
        plt.ylabel('Y', fontsize=12)
        plt.title(f'Твой график ({len(x)} точек)', fontsize=14)
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=10)
        plt.tight_layout()

        bio = BytesIO()
        plt.savefig(bio, format='png', dpi=150)
        bio.seek(0)
        plt.close()

        # Формируем подпись с угловым коэффициентом
        caption = (f"📊 *Твой график*\n"
                  f"Точек: {len(x)}\n"
                  f"Среднее X: {np.mean(x):.2f}\n"
                  f"Среднее Y: {np.mean(y):.2f}\n\n"
                  f"📐 *Угловой коэффициент:*\n"
                  f"k = {k:.2f}\n"
                  f"(при увеличении X на 1, Y меняется на {k:.2f})\n\n"
                  f"📈 *Свободный член:* b = {b:.2f}")

        if len(x) >= 3:
            caption += f"\n\n🔵 *Квадратичная:* a={a:.2f}, b={b2:.2f}, c={c:.2f}"

        await update.message.reply_photo(bio, caption=caption, parse_mode='Markdown')

        # Сбрасываем данные и возвращаем в меню
        user_mode[uid] = "menu"
        user_graph_data[uid] = {'x': [], 'y': []}
        await update.message.reply_text("✅ *График готов!* Что дальше?", reply_markup=MAIN_KB, parse_mode='Markdown')
        return

    # Проверяем, не кнопка ли меню
    if txt in ["🔙 меню", "меню"]:
        user_mode[uid] = "menu"
        user_graph_data[uid] = {'x': [], 'y': []}
        await update.message.reply_text("*🏠 Главное меню*", reply_markup=MAIN_KB, parse_mode='Markdown')
        return

    # Парсим числа
    try:
        n = [float(x) for x in txt.split()]
        if len(n) % 2 != 0:
            await update.message.reply_text("❌ Введи чётное количество чисел (пары x y)!\nПример: `1 2.1 2 3.9 3 6.2`")
            return

        user_graph_data[uid]['x'].extend(n[::2])
        user_graph_data[uid]['y'].extend(n[1::2])

        total = len(user_graph_data[uid]['x'])
        await update.message.reply_text(
            f"✅ Добавлено {len(n)//2} точек. Всего: {total}\n"
            "Продолжай или напиши `Готово`\n"
            "Для выхода нажми `🔙 Меню`"
        )
    except:
        await update.message.reply_text("❌ Введи числа через пробел!\nПример: `1 2 3 4 5 6`")

print("✅ График готов!")

✅ График готов!


In [8]:
async def main():
    app = Application.builder().token(TOKEN).build()
    app.add_handler(CommandHandler('start', start))
    app.add_handler(CallbackQueryHandler(exp_info, pattern='^(g|pend|lens|dens)$'))
    app.add_handler(CallbackQueryHandler(dens_info, pattern='^(d_f|d_v|d_l)$'))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_msg))

    print("\n✅ БОТ ЗАПУЩЕН! Напиши /start\n")
    await app.run_polling(drop_pending_updates=True)

loop = asyncio.get_event_loop()
loop.create_task(main())

<Task pending name='Task-1' coro=<main() running at /tmp/ipykernel_39319/1714081693.py:1>>